Comparative experiment: A comparison of traditional hair removal algorithms (DullRazor) and my method

In [1]:
import os, cv2, torch, re
import numpy as np
from tqdm import tqdm
from models.UltraLight_VM_UNet import UltraLight_VM_UNet
from sklearn.metrics import confusion_matrix

# 1. Path configuration
IMG_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/dermoscopic_image"
MASK_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/lesion_mask"
WEIGHT_PATH = "/gpfs/work/bio/yixuanli2204/FYP_Project/code/UltraLight-VM-UNet/results/UltraLight_VM_UNet_ISIC_Combined_20260323_best/checkpoints/best.pth"
SAVE_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/restored_dullrazor"
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = torch.device("cuda")

# 2. DullRazor algorithm implementation
def apply_dull_razor(img):
    # Convert to grayscale
    grayScale = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Black hat filter (extract dark lines)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9,9)) 
    blackhat = cv2.morphologyEx(grayScale, cv2.MORPH_BLACKHAT, kernel)
    # Gaussian filter (smooth noise)
    bhg = cv2.GaussianBlur(blackhat, (3,3), cv2.BORDER_DEFAULT)
    # Binary thresholding (generate hair mask)
    ret, mask = cv2.threshold(bhg, 10, 255, cv2.THRESH_BINARY)
    # Image inpainting (fill masked regions using Telea algorithm)
    dst = cv2.inpaint(img, mask, 6, cv2.INPAINT_TELEA)
    return dst

# 3. Main evaluation logic
def run_comparison():
    model = UltraLight_VM_UNet(num_classes=1, c_list=[8,16,24,32,48,64], bridge=True).to(DEVICE)
    model.load_state_dict(torch.load(WEIGHT_PATH, map_location=DEVICE))
    model.eval()

    img_files = [f for f in os.listdir(IMG_DIR) if f.endswith('.png')]
    all_cms = []

    print(f"Running DullRazor comparison experiment (samples: {len(img_files)})...")

    for fname in tqdm(img_files):
        # A. Read original image
        raw_img = cv2.imread(os.path.join(IMG_DIR, fname))
        
        # B. Apply DullRazor preprocessing
        dull_img = apply_dull_razor(raw_img)
        # cv2.imwrite(os.path.join(SAVE_DIR, fname), dull_img) # Uncomment to save images
        
        # C. Prepare VM-UNet input (256x256)
        img_rgb = cv2.cvtColor(dull_img, cv2.COLOR_BGR2RGB)
        img_t = torch.from_numpy(cv2.resize(img_rgb, (256, 256))).permute(2,0,1).unsqueeze(0).float().to(DEVICE) / 255.0
        
        # D. Read ground truth
        match = re.search(r"ISIC_(\d{7})", fname)
        img_id = match.group(1)
        gt = cv2.imread(os.path.join(MASK_DIR, f"ISIC_{img_id}_lesion.png"), 0)
        if gt is None: 
            continue
        gt_256 = (cv2.resize(gt, (256, 256)) > 127).astype(np.uint8)

        # E. Prediction
        with torch.no_grad():
            output = model(img_t)
            pred = (output.squeeze().cpu().numpy() > 0.5).astype(np.uint8)

        # F. Statistics
        cm = confusion_matrix(gt_256.flatten(), pred.flatten(), labels=[0, 1])
        all_cms.append(cm)

    # Calculate final scores
    total_cm = sum(all_cms)
    tn, fp, fn, tp = total_cm.ravel()
    dice = (2 * tp) / (2 * tp + fp + fn + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)

    print(f"DullRazor + VM-UNet evaluation results:")
    print(f"Dice Score: {dice:.4f}")
    print(f"mIoU:       {iou:.4f}")

if __name__ == "__main__":
    run_comparison()

/gpfs/work/bio/yixuanli2204/.conda/envs/fyp/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SC_Att_Bridge was used
Running DullRazor comparison experiment (samples: 500)...


100%|██████████| 500/500 [01:51<00:00,  4.47it/s]

DullRazor + VM-UNet evaluation results:
Dice Score: 0.8020
mIoU:       0.6695
